# 案例4：起点中文网原创风云榜爬取

爬取起点中文网原创风云榜前 5 页的数据，提取书名、作者、简介和更新日期，并保存为 CSV 文件。

## 一、导入库

使用 Requests 获取网页，使用 lxml 和 XPath 解析 HTML，使用 csv 模块保存数据。

In [ ]:
%pip install requests lxml

In [ ]:
import requests
from lxml import etree
import csv

## 二、获取网页

`get_html()` 发送 GET 请求，自动识别网页编码，并检查请求状态。

In [ ]:
def get_html(url, time=30):
    try:
        r = requests.get(url, timeout=time)
        r.encoding = r.apparent_encoding
        r.raise_for_status()
        return r.text
    except Exception as error:
        print(error)

## 三、解析书籍信息

根据教材中的页面结构，书籍信息位于 `book-img-text` 下的 `book-mid-info` 区域，分别提取书名、作者、简介和更新日期。

In [ ]:
def parser(html):
    doc = etree.HTML(html)
    out_list = []

    for row in doc.xpath("//div[@class='book-img-text']//li/div[@class='book-mid-info']"):
        row_data = [
            row.xpath("h4/a/text()")[0],
            row.xpath("p[@class='author']/a/text()")[0],
            row.xpath("p[2]/text()")[0].strip(),
            row.xpath("p[@class='update']/span/text()")[0]
        ]
        out_list.append(row_data)

    return out_list

## 四、保存为 CSV 文件

使用 `csv.writer()` 将二维列表写入 CSV 文件，并添加字段名称。

In [ ]:
def save_csv(item, path):
    with open(path, "w", newline="", encoding="utf-8-sig") as f:
        csv_write = csv.writer(f)
        csv_write.writerow(["书名", "作者", "简介", "更新日期"])
        csv_write.writerows(item)

## 五、分页爬取

观察教材中的分页规律，页面地址通过修改 `page` 参数实现。这里循环爬取第 1 页至第 5 页，并将各页结果合并。

In [ ]:
all_list = []

for i in range(1, 6):
    url = "http://www.bspider.top/qidian/?page={}".format(i)
    html = get_html(url)
    page_list = parser(html)
    all_list.extend(page_list)
    print("第", i, "页爬取数量：", len(page_list))

save_csv(all_list, "qidian.csv")
print("共爬取书籍：", len(all_list), "条")